In [4]:
# Imports
import os
import requests 
from bs4 import BeautifulSoup
from typing import List
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.schema import AIMessage, SystemMessage, HumanMessage

import gradio as gr

In [5]:
# Environment variables
load_dotenv()

os.environ['GOOGLE_API_KEY'] = os.getenv('GOOGLE_API_KEY')

In [6]:
model = "gemini-2.0-flash-lite"
gemini = ChatGoogleGenerativeAI(
    model = model,
    temperature = 0.4
)

In [7]:
# A generic system message
system_prompt = "You are a helpful assistant."

In [10]:
def message_gemini(prompt):
    
    messages = [
        SystemMessage(content = system_prompt),
        HumanMessage(content = prompt)
    ]
    
    response = gemini.invoke(messages)
    return response.content

In [11]:
message_gemini("What is today's date?")

'Today is October 26, 2023.'

## User Interface!!!

In [12]:
# A simple function
def shout(text):
    return text.upper()

In [13]:
shout("hello")

'HELLO'

In [14]:
view = gr.Interface(fn=shout, inputs="textbox", outputs="textbox")
view.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Created dataset file at: .gradio/flagged/dataset1.csv


In [16]:
view = gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode = "never")
view.launch(share=True)

* Running on local URL:  http://127.0.0.1:7862

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2025/09/26 23:05:05 [W] [service.go:132] login to server failed: i/o deadline reached


In [17]:
view = gr.Interface(
    fn=shout,
    inputs=[gr.Textbox(label="Your Messages: ", lines=6)],
    outputs=[gr.Textbox(label="Response: ", lines=8)],
    flagging_mode="never"
)

view.launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


In [18]:
view = gr.Interface(
    fn=message_gemini,
    inputs=[gr.Textbox(label="Your Messages: ", lines=6)],
    outputs=[gr.Textbox(label="Response: ", lines=8)],
    flagging_mode="never"
)

view.launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


In [20]:
system_message = "You are a helpful assistant respond in markdown."

view = gr.Interface(
    fn=message_gemini,
    inputs=[gr.Textbox(label="Your Messages: ")],
    outputs=[gr.Markdown(label="Response: ")],
    flagging_mode="never"
)

view.launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


In [34]:
def stream_gemini(prompt):
    
    messages = [
        SystemMessage(content=system_message),
        HumanMessage(content=prompt)
    ]

    response_text = ""

    for chunk in gemini.stream(messages):
        response_text += chunk.content
        yield response_text # stream progressively
    

In [36]:
view = gr.Interface(
    fn=stream_gemini,
    inputs = [gr.Textbox(label="Your Messages: ")],
    outputs = [gr.Markdown(label="Response: ")],
    flagging_mode = "never"
)

view.launch()

* Running on local URL:  http://127.0.0.1:7872
* To create a public link, set `share=True` in `launch()`.


In [38]:
# Initializing the second model
model = "gemini-2.5-flash"
gemini2 = ChatGoogleGenerativeAI(
    model = model,
    temperature = 0.4
)

In [39]:
# Streaming 2nd model
def stream_gemini2(prompt):
    
    messages = [
        SystemMessage(content=system_message),
        HumanMessage(content=prompt)
    ]

    response_text = ""

    for chunk in gemini.stream(messages):
        response_text += chunk.content
        yield response_text # stream progressively
    

In [40]:
def stream_model(prompt, model):
    if model == "Gemini":
        result = stream_gemini(prompt)
    if model == "Gemini2":
        result = stream_gemini2(prompt)
    else:
        raise ValueError("Unknown Model")

    for chunk in result:
        yield chunk

In [47]:
view = gr.Interface(
    fn = stream_model,
    inputs = [gr.Textbox(label="Your Message: "),
              gr.Dropdown(["Gemini", "Gemini2"],
              label = "Select Model")],
    outputs = [gr.Markdown(label="Response")],
    flagging_mode = "never"
) 
view.launch()


* Running on local URL:  http://127.0.0.1:7874
* To create a public link, set `share=True` in `launch()`.
